# Neurality: Session-Based Adaptive Tag Affinity & Negative Interest Memory
This notebook implements positive interest memory and negative feedback memory (skips, low watch time) to dynamically shape the user's tag affinity vector.

In [1]:
# ==================================================
# NOTEBOOK VALIDATION & DEPENDENCY VERIFICATION PIPELINE
# ==================================================
import sys
import time
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import seaborn as sns
import torch

print(f"[SUCCESS] Jupyter Kernel Python Version: {sys.version}")
print(f"numpy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scipy: {scipy.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"seaborn: {sns.__version__}")
print(f"torch: {torch.__version__}")
print("[HEALTH CHECK] Conda kernel detection and package imports are 100% stable!")


[SUCCESS] Jupyter Kernel Python Version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]
numpy: 2.1.3
pandas: 2.2.3
scipy: 1.15.3
scikit-learn: 1.6.1
matplotlib: 3.10.0
seaborn: 0.13.2
torch: 2.10.0+cpu
[HEALTH CHECK] Conda kernel detection and package imports are 100% stable!


In [2]:
# Dynamic Tag Affinity Update Logic
from collections import defaultdict

class TagAffinityModel:
    def __init__(self, decay_rate=0.90):
        self.positive_affinities = defaultdict(float)
        self.negative_affinities = defaultdict(float)
        self.decay_rate = decay_rate

    def update_behavior(self, tags, watch_time, duration, completed, liked=False, saved=False):
        watch_ratio = watch_time / duration if duration > 0 else 0
        is_skip = watch_time < 2.0

        # Apply recency decay to all existing weights first
        for tag in list(self.positive_affinities.keys()):
            self.positive_affinities[tag] *= self.decay_rate
        for tag in list(self.negative_affinities.keys()):
            self.negative_affinities[tag] *= self.decay_rate

        # Calculate rewards
        if is_skip:
            # Skip Penalty
            for tag in tags:
                self.negative_affinities[tag.lower()] += 1.5
        else:
            # Positive Interaction reward
            reward = watch_ratio * 1.0
            if completed:
                reward += 1.5
            if liked:
                reward += 2.0
            if saved:
                reward += 2.5

            for tag in tags:
                self.positive_affinities[tag.lower()] += reward

    def get_affinity_score(self, tag):
        tag_l = tag.lower()
        return self.positive_affinities[tag_l] - self.negative_affinities[tag_l]

model = TagAffinityModel()
print("Initialized TagAffinityModel.")

Initialized TagAffinityModel.


In [3]:
# Simulate User Session Binge and Interest Drift
model = TagAffinityModel(decay_rate=0.95)

# Sequence of 15 views: User binges #phonk & #edits, but skips #crypto and #cringe memes
history = [
    # Phonk binging
    (['phonk', 'edits'], 15.0, 15.0, True, True, False), # Complete & like
    (['phonk', 'dark'], 15.0, 15.0, True, False, True), # Complete & save
    # Skips crypto spam
    (['crypto', 'investing'], 1.2, 15.0, False, False, False), # Skip
    (['crypto', 'cringe'], 0.8, 15.0, False, False, False), # Skip
    # Back to phonk / gaming
    (['phonk', 'gaming'], 12.0, 15.0, False, False, False), # High watch time
    (['gaming', 'tech'], 15.0, 15.0, True, True, False), # Complete & like
]

for idx, (tags, wt, dur, comp, lk, sv) in enumerate(history):
    model.update_behavior(tags, wt, dur, comp, lk, sv)
    print(f"View {idx+1}: Tags={tags} | watch={wt}s | comp={comp}")
    print(f"  Affinity phonk: {model.get_affinity_score('phonk'):.2f}")
    print(f"  Affinity crypto: {model.get_affinity_score('crypto'):.2f}")

View 1: Tags=['phonk', 'edits'] | watch=15.0s | comp=True
  Affinity phonk: 4.50
  Affinity crypto: 0.00
View 2: Tags=['phonk', 'dark'] | watch=15.0s | comp=True
  Affinity phonk: 9.27
  Affinity crypto: 0.00
View 3: Tags=['crypto', 'investing'] | watch=1.2s | comp=False
  Affinity phonk: 8.81
  Affinity crypto: -1.50
View 4: Tags=['crypto', 'cringe'] | watch=0.8s | comp=False
  Affinity phonk: 8.37
  Affinity crypto: -2.92
View 5: Tags=['phonk', 'gaming'] | watch=12.0s | comp=False
  Affinity phonk: 8.75
  Affinity crypto: -2.78
View 6: Tags=['gaming', 'tech'] | watch=15.0s | comp=True
  Affinity phonk: 8.31
  Affinity crypto: -2.64


In [4]:
# Visualizing Tag Affinity Shift
all_tags = ['phonk', 'edits', 'dark', 'crypto', 'cringe', 'gaming', 'tech', 'investing']
scores = [model.get_affinity_score(t) for t in all_tags]

plt.figure(figsize=(8, 4))
sns.barplot(x=scores, y=all_tags, palette='coolwarm')
plt.axvline(0, color='red', linestyle='--')
plt.title('Dynamic User Tag Affinity Map (Positive vs Negative Memory)')
plt.xlabel('Aflatility Score')
plt.ylabel('Tags')
plt.show()

NameError: name 'plt' is not defined